# Turkey investing notebook

This notebook tests monthly salary investing over the last months with Turkey focused assets.


In [ ]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn


In [ ]:

import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf
import plotly.express as px
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier


In [ ]:

CONFIG = {
    "tickers": ["THYAO.IS", "AKBNK.IS", "EREGL.IS", "ZGOLD.IS", "USDTRY=X"],
    "labels": {
        "THYAO.IS": "THYAO",
        "AKBNK.IS": "AKBNK",
        "EREGL.IS": "EREGL",
        "ZGOLD.IS": "GOLD",
        "USDTRY=X": "USDTRY",
    },
    "salary_try": 25000,
    "test_months": 6,
    "lookback_days": 252,
    "download_period": "10y",
    "max_weight": 0.70,
    "gold_label": "GOLD",
    "gold_min": 0.10,
    "gold_max": 0.55,
}
CONFIG


In [ ]:

def try_fmt(x):
    if pd.isna(x):
        return "-"
    return f"₺{x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

def pct_fmt(x):
    if pd.isna(x):
        return "-"
    return f"{x*100:.2f}%"

def style(fig, title):
    fig.update_layout(template="plotly_white", title=title, hovermode="x unified", legend_title_text="")
    return fig

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return w / w.sum()
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return w / w.sum()
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return w / w.sum()

def views_from_prices(price_window):
    rets = price_window.pct_change().dropna()
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    return score.clip(-0.15, 0.15).to_dict()


In [ ]:

rows = []
series = []
for t in CONFIG["tickers"]:
    df = yf.download(t, period=CONFIG["download_period"], auto_adjust=True, progress=False)
    if len(df) == 0:
        rows.append({"ticker": t, "label": CONFIG["labels"][t], "n": 0, "status": "empty"})
        continue
    s = df["Close"] if "Close" in df.columns else df.iloc[:, 0]
    s = pd.Series(s).dropna()
    s.name = CONFIG["labels"][t]
    rows.append({"ticker": t, "label": CONFIG["labels"][t], "n": len(s), "start": str(s.index.min().date()), "end": str(s.index.max().date()), "status": "ok"})
    series.append(s)
coverage = pd.DataFrame(rows)
display(coverage)
prices = pd.concat(series, axis=1).sort_index().ffill()
firsts = prices.apply(lambda s: s.first_valid_index())
prices = prices.loc[firsts.max():].dropna()
lookback = min(CONFIG["lookback_days"], max(60, len(prices) - 40))
months = min(CONFIG["test_months"], max(3, len(prices.resample("MS").first().dropna()) - 1))
print("assets", list(prices.columns))
print("days", len(prices), "lookback", lookback, "test_months", months)
display(prices.tail())


In [ ]:

fig = px.line(prices / prices.iloc[0] * 100, x=prices.index, y=prices.columns)
style(fig, "Normalized prices").show()
fig = px.imshow(prices.pct_change().dropna().corr(), text_auto=True, aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
style(fig, "Correlation").show()


In [ ]:

def ew_weights(cols):
    return pd.Series(1 / len(cols), index=cols)

def hrp_weights(price_window):
    hrp = HRPOpt(price_window.pct_change().dropna())
    w = pd.Series(hrp.optimize()).reindex(price_window.columns).fillna(0.0)
    return w / w.sum()

def bl_weights(price_window):
    S = CovarianceShrinkage(price_window).ledoit_wolf()
    views = views_from_prices(price_window)
    bl = BlackLittermanModel(S, pi="equal", absolute_views=views, omega="default")
    post = bl.bl_returns()
    ef = EfficientFrontier(post, S, weight_bounds=(0, CONFIG["max_weight"]))
    ef.max_sharpe(risk_free_rate=0.0)
    w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
    w = w / w.sum()
    return band_gold(w, CONFIG["gold_label"], CONFIG["gold_min"], CONFIG["gold_max"])

def run_strategy(name):
    monthly = prices.resample("MS").first().dropna()
    rebalance_dates = monthly.index[-months:]
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs = [], []
    for i, dt in enumerate(rebalance_dates):
        cash += CONFIG["salary_try"]
        window = prices.loc[:dt].tail(lookback)
        if name == "Equal":
            w = ew_weights(prices.columns)
        elif name == "HRP":
            w = hrp_weights(window)
        else:
            w = bl_weights(window)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (w * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)
        for t, v in vals.items():
            hist.append({"date": t, "strategy": name, "portfolio_value": float(v)})
        row = {"date": dt, "strategy": name, "capital_after_contribution": total}
        for c, x in w.items():
            row[f"weight_{c}"] = float(x)
        rebs.append(row)
    return pd.DataFrame(hist).drop_duplicates(["date", "strategy"]), pd.DataFrame(rebs)

parts_h, parts_w = [], []
for s in ["Equal", "HRP", "BL"]:
    h, w = run_strategy(s)
    parts_h.append(h)
    parts_w.append(w)
hist_df = pd.concat(parts_h, ignore_index=True)
weights_df = pd.concat(parts_w, ignore_index=True)
display(hist_df.tail())


In [ ]:

rows = []
contributed = CONFIG["salary_try"] * months
for s in ["Equal", "HRP", "BL"]:
    sub = hist_df[hist_df["strategy"] == s].sort_values("date").copy()
    sub["ret"] = sub["portfolio_value"].pct_change().fillna(0.0)
    dd = sub["portfolio_value"] / sub["portfolio_value"].cummax() - 1
    rows.append({
        "Strategy": s,
        "Contributed": try_fmt(contributed),
        "Ending Value": try_fmt(sub["portfolio_value"].iloc[-1]),
        "Net Profit": try_fmt(sub["portfolio_value"].iloc[-1] - contributed),
        "TWR": pct_fmt(sub["portfolio_value"].iloc[-1] / contributed - 1),
        "Ann Vol": pct_fmt(sub["ret"].std() * np.sqrt(252)),
        "Sharpe": f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
        "MaxDD": pct_fmt(dd.min()),
    })
summary = pd.DataFrame(rows)
display(summary)
fig = px.line(hist_df, x="date", y="portfolio_value", color="strategy")
fig.update_yaxes(tickprefix="₺", separatethousands=True)
style(fig, "Portfolio value").show()


In [ ]:

last_weights = weights_df.sort_values("date").groupby("strategy").tail(1).copy()
cols = [c for c in last_weights.columns if c.startswith("weight_")]
out = last_weights[["strategy"] + cols].copy()
out.columns = ["Strategy"] + [c.replace("weight_", "") for c in cols]
for c in out.columns[1:]:
    out[c] = out[c].map(pct_fmt)
display(out.reset_index(drop=True))
